# CIFAR-100: Spectral Pruning via AEI — Comparison with Other Methods

**Option 3 Experiment:** Same SimpleCNN architecture as CIFAR-10 paper experiment,
but trained on CIFAR-100 (100 classes) to address Reviewer 2's concern about model/task complexity.

**Architecture:** Conv1(3→32) → Conv2(32→64) → FC1(4096→256) → FC2(256→**100**)

**Methods compared:** Spectral (AEI), Hybrid (AEI×L2), L1 Norm, L2 Norm, SNIP, GraSP, Random

**Sparsity levels:** 20%, 30%, 40%

**Pruning target:** Conv2 filters (same as CIFAR-10 experiment)


## 0. Mount Drive & Install Dependencies

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/AEI_Experiments/CIFAR100'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save directory: {SAVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save directory: /content/drive/MyDrive/AEI_Experiments/CIFAR100


In [15]:
!pip install scipy numpy matplotlib seaborn torch torchvision -q

## 1. Imports

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
import scipy
from scipy.linalg import eigh
from scipy.sparse.linalg import eigsh
from scipy.stats import pearsonr, spearmanr

import copy
import time
import random
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import defaultdict

# ── reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 2. CIFAR-100 Data Loading

In [17]:
# ── CIFAR-100 has same image size as CIFAR-10 (32×32 RGB)
# but 100 classes instead of 10

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),   # CIFAR-100 mean
                         (0.2675, 0.2565, 0.2761)),   # CIFAR-100 std
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])

trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
testloader  = torch.utils.data.DataLoader(
    testset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Subset loader for activation collection (same as CIFAR-10 experiment)
subset_indices = list(range(5000))
subset_dataset = torch.utils.data.Subset(trainset, subset_indices)
activationloader = torch.utils.data.DataLoader(
    subset_dataset, batch_size=256, shuffle=False, num_workers=2)

NUM_CLASSES = 100
print(f'Train size: {len(trainset)}, Test size: {len(testset)}')
print(f'Number of classes: {NUM_CLASSES}')

Train size: 50000, Test size: 10000
Number of classes: 100


## 3. Model Architecture

**Identical to CIFAR-10 experiment** except FC2 output is 100 instead of 10.

In [18]:
class SimpleCNN(nn.Module):
    """
    Same architecture as the CIFAR-10 experiment in the paper:
      Conv1: 3  → 32 filters, 3×3, ReLU, MaxPool 2×2
      Conv2: 32 → 64 filters, 3×3, ReLU, MaxPool 2×2
      FC1:   4096 → 256
      FC2:   256  → num_classes (100 for CIFAR-100)
    """
    def __init__(self, num_classes=100, conv2_filters=64):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, conv2_filters, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.relu  = nn.ReLU()

        # After two MaxPool(2,2): 32×32 → 8×8
        self.fc_input_size = conv2_filters * 8 * 8
        self.fc1 = nn.Linear(self.fc_input_size, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 32×32 → 16×16
        x = self.pool(self.relu(self.conv2(x)))   # 16×16 →  8×8
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_flops(model, input_size=(1, 3, 32, 32)):
    """Approximate FLOPs for the SimpleCNN."""
    # Conv1: 3×32×32 * 32 * 3×3 * 2 = 2 * 32 * 3 * 32 * 32 * 9
    conv1_flops = 2 * 32 * 3 * 32 * 32 * 9
    # Conv2: 32×16×16 * F * 3×3 * 2
    F = model.conv2.out_channels
    conv2_flops = 2 * F * 32 * 16 * 16 * 9
    # FC1: 2 * F*64 * 256
    fc1_flops = 2 * F * 64 * 256
    # FC2: 2 * 256 * num_classes
    fc2_flops = 2 * 256 * model.fc2.out_features
    total = conv1_flops + conv2_flops + fc1_flops + fc2_flops
    return total


# Baseline model
baseline_model = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)
print(f'Baseline parameters: {count_parameters(baseline_model):,}')
print(f'Baseline FLOPs: {count_flops(baseline_model)/1e6:.2f}M')

Baseline parameters: 1,093,924
Baseline FLOPs: 13.36M


## 4. Training & Evaluation Utilities

In [22]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(targets).sum().item()
        total   += inputs.size(0)
    return total_loss / total, 100. * correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            correct += predicted.eq(targets).sum().item()
            total   += inputs.size(0)
    return 100. * correct / total


def train_model(model, epochs=50, lr=1e-3, finetune=False, verbose=True):
    """
    Full training or fine-tuning.
    finetune=True uses lr=1e-4 and fewer epochs (same as paper).
    """
    if finetune:
        lr = 1e-4
        epochs = 20      # same fine-tune budget as CIFAR-10 experiment

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    history  = []
    for epoch in range(epochs):
        loss, train_acc = train_epoch(model, trainloader, optimizer, criterion)
        test_acc        = evaluate(model, testloader)
        scheduler.step()
        history.append({'epoch': epoch+1, 'loss': loss,
                        'train_acc': train_acc, 'test_acc': test_acc})
        if test_acc > best_acc:
            best_acc = test_acc
        if verbose and (epoch+1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | '
                  f'Loss: {loss:.4f} | Train: {train_acc:.2f}% | '
                  f'Test: {test_acc:.2f}%')
    return best_acc, history

## 5. Baseline Training

In [20]:
print('='*60)
print('TRAINING BASELINE SimpleCNN on CIFAR-100')
print('='*60)

baseline_model = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)

# CIFAR-100 is harder — train for more epochs than CIFAR-10
TRAIN_EPOCHS = 80

t0 = time.time()
baseline_acc, history = train_model(baseline_model, epochs=TRAIN_EPOCHS,
                                     lr=1e-3, verbose=True)
train_time = time.time() - t0

baseline_acc = evaluate(baseline_model, testloader)
print(f'\nBaseline Accuracy: {baseline_acc:.2f}%')
print(f'Training time: {train_time/60:.1f} min')
print(f'Baseline FLOPs: {count_flops(baseline_model)/1e6:.2f}M')

# Save baseline
torch.save(baseline_model.state_dict(),
           os.path.join(SAVE_DIR, 'baseline_cifar100.pth'))
print('Baseline model saved.')

TRAINING BASELINE SimpleCNN on CIFAR-100
  Epoch  10/80 | Loss: 2.2510 | Train: 41.06% | Test: 42.11%
  Epoch  20/80 | Loss: 1.9663 | Train: 47.49% | Test: 46.33%
  Epoch  30/80 | Loss: 1.8108 | Train: 50.82% | Test: 48.59%
  Epoch  40/80 | Loss: 1.7033 | Train: 53.47% | Test: 49.46%
  Epoch  50/80 | Loss: 1.6175 | Train: 55.65% | Test: 50.43%
  Epoch  60/80 | Loss: 1.5495 | Train: 57.24% | Test: 50.70%
  Epoch  70/80 | Loss: 1.5093 | Train: 58.27% | Test: 51.27%
  Epoch  80/80 | Loss: 1.4941 | Train: 58.78% | Test: 51.43%

Baseline Accuracy: 51.43%
Training time: 30.3 min
Baseline FLOPs: 13.36M
Baseline model saved.


## 6. AEI Core Functions

Identical logic to CIFAR-10 experiment — spatial global-average pooling
to reduce conv feature maps to per-filter scalars, then Pearson correlation graph.

In [23]:
def collect_conv2_activations(model, loader):
    """
    Collect Conv2 filter activations via spatial global-average pooling.
    Returns array of shape [N_samples, F] where F = number of conv2 filters.
    """
    model.eval()
    activations = []

    def hook_fn(module, input, output):
        # output shape: [B, F, H, W] — apply global average pooling
        pooled = output.mean(dim=[2, 3])  # [B, F]
        activations.append(pooled.detach().cpu())

    hook = model.conv2.register_forward_hook(hook_fn)

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(DEVICE)
            model(inputs)

    hook.remove()
    return torch.cat(activations, dim=0).numpy()  # [N, F]


def build_coactivation_graph(activations):
    """
    Build Pearson correlation co-activation graph.
    Returns adjacency matrix A of shape [F, F] with non-negative weights.
    """
    F = activations.shape[1]
    A = np.zeros((F, F))
    for i in range(F):
        for j in range(i+1, F):
            r, _ = pearsonr(activations[:, i], activations[:, j])
            if np.isnan(r):
                r = 0.0
            w = abs(r)   # absolute correlation as edge weight
            A[i, j] = w
            A[j, i] = w
    return A


def compute_fiedler_vector(A):
    """
    Compute Graph Laplacian and extract Fiedler vector (v2).
    """
    D = np.diag(A.sum(axis=1))
    L = D - A
    # Only need 2nd smallest eigenvalue/vector
    eigenvalues, eigenvectors = eigh(L)
    # eigenvalues sorted ascending; index 0 is λ1≈0, index 1 is λ2 (Fiedler)
    fiedler_vector = eigenvectors[:, 1]
    return fiedler_vector, eigenvalues[1]


def compute_aei_scores(A, fiedler_vector):
    """
    Compute AEI score Ri = Σ_{(i,j)∈Ei} |v2,i - v2,j| for each node i.
    """
    F = A.shape[0]
    R = np.zeros(F)
    for i in range(F):
        neighbors = np.where(A[i] > 0)[0]
        R[i] = sum(abs(fiedler_vector[i] - fiedler_vector[j])
                   for j in neighbors)
    return R


def get_aei_scores(model, loader):
    """
    Full pipeline: activations → graph → Fiedler → AEI scores.
    Returns R scores and adjacency matrix.
    """
    t0 = time.time()
    acts = collect_conv2_activations(model, loader)
    t_act = time.time() - t0

    t1 = time.time()
    A = build_coactivation_graph(acts)
    t_graph = time.time() - t1

    t2 = time.time()
    fiedler, lambda2 = compute_fiedler_vector(A)
    t_eig = time.time() - t2

    R = compute_aei_scores(A, fiedler)

    overhead = {
        'act_collect_ms': t_act * 1000,
        'graph_ms':       t_graph * 1000,
        'eig_ms':         t_eig * 1000,
        'total_ms':       (time.time() - t0) * 1000,
    }
    return R, A, fiedler, overhead


print('AEI functions defined.')

AEI functions defined.


## 7. Pruning Methods

In [28]:
def minmax_normalize(arr):
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-12:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)


# ── Saliency scorers ─────────────────────────────────────────────────────────

def score_l1(model):
    """L1 norm of each Conv2 filter."""
    w = model.conv2.weight.data.cpu().numpy()  # [F, 32, 3, 3]
    return np.abs(w).sum(axis=(1, 2, 3))


def score_l2(model):
    """L2 norm of each Conv2 filter."""
    w = model.conv2.weight.data.cpu().numpy()
    return np.sqrt((w**2).sum(axis=(1, 2, 3)))


def score_snip(model, loader, criterion=None, n_batches=5):
    """
    SNIP: connection sensitivity = |gradient × weight| summed per filter.
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    model.eval()
    # Zero existing gradients
    model.zero_grad()
    total_grad = None

    for i, (inputs, targets) in enumerate(loader):
        if i >= n_batches:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

    w    = model.conv2.weight.data.cpu().numpy()
    grad = model.conv2.weight.grad.data.cpu().numpy()
    sensitivity = np.abs(grad * w).sum(axis=(1, 2, 3))
    model.zero_grad()
    return sensitivity


def score_grasp(model, loader, criterion=None, n_batches=5):
    """
    GraSP: gradient signal preservation = -Hg ⊙ w summed per filter.
    Uses gradient of gradient norm as Hessian-gradient product approximation.
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    model.eval()
    model.zero_grad()

    # Pass 1: compute full gradient
    for i, (inputs, targets) in enumerate(loader):
        if i >= n_batches:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

    grad1 = model.conv2.weight.grad.data.clone()
    model.zero_grad()

    # Pass 2: compute gradient of gradient norm (Hessian-gradient product)
    for i, (inputs, targets) in enumerate(loader):
        if i >= n_batches:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        grad_norm = torch.autograd.grad(
            loss, model.conv2.weight,
            retain_graph=True, create_graph=True)[0]
        (grad_norm * grad1.to(DEVICE)).sum().backward()

    Hg = model.conv2.weight.grad.data.cpu().numpy()
    w  = model.conv2.weight.data.cpu().numpy()
    score = -(Hg * w).sum(axis=(1, 2, 3))
    model.zero_grad()
    return score


def score_hybrid(R_scores, model):
    """Hybrid (AEI × L2): geometric mean of normalized AEI and L2 scores."""
    l2  = score_l2(model)
    aei = R_scores
    return minmax_normalize(aei) * minmax_normalize(l2)


def score_random(model):
    """Random scores (for baseline comparison)."""
    F = model.conv2.out_channels
    return np.random.rand(F)


print('Scoring functions defined.')

Scoring functions defined.


In [29]:
# ── Pruning & Fine-tuning ────────────────────────────────────────────────────

def prune_model(model_original, scores, sparsity, num_classes=100):
    """
    Structured pruning of Conv2 filters.
    Keeps filters with the HIGHEST scores (removes lowest-scoring ones).
    Updates conv2 and fc1 accordingly.
    """
    F_orig = model_original.conv2.out_channels
    n_keep = max(1, int(F_orig * (1.0 - sparsity)))

    # Indices of filters to KEEP (highest scores)
    keep_idx = np.argsort(scores)[-n_keep:]  # ascending sort, take last n_keep
    keep_idx = np.sort(keep_idx)

    # Build pruned model
    pruned_model = SimpleCNN(num_classes=num_classes,
                             conv2_filters=n_keep).to(DEVICE)

    # Copy conv1 weights (unchanged)
    pruned_model.conv1.weight.data = \
        model_original.conv1.weight.data.clone()
    pruned_model.conv1.bias.data = \
        model_original.conv1.bias.data.clone()

    # Copy selected conv2 filters
    pruned_model.conv2.weight.data = \
        model_original.conv2.weight.data[keep_idx].clone()
    pruned_model.conv2.bias.data = \
        model_original.conv2.bias.data[keep_idx].clone()

    # Update fc1: only keep input channels corresponding to kept filters
    # Each filter contributes 8*8 = 64 input neurons to fc1
    spatial = 8 * 8
    fc1_keep_idx = np.concatenate(
        [np.arange(k * spatial, (k+1) * spatial) for k in keep_idx])
    pruned_model.fc1.weight.data = \
        model_original.fc1.weight.data[:, fc1_keep_idx].clone()
    pruned_model.fc1.bias.data = \
        model_original.fc1.bias.data.clone()

    # Copy fc2 (unchanged)
    pruned_model.fc2.weight.data = \
        model_original.fc2.weight.data.clone()
    pruned_model.fc2.bias.data = \
        model_original.fc2.bias.data.clone()

    return pruned_model, keep_idx


def prune_and_finetune(model_original, scores, sparsity,
                        num_classes=100, finetune_epochs=20):
    """
    Full prune + fine-tune pipeline.
    Returns post-finetune accuracy, pruned model, and kept filter indices.
    """
    pruned_model, keep_idx = prune_model(
        model_original, scores, sparsity, num_classes)

    acc_after_pruning = evaluate(pruned_model, testloader)

    # Fine-tune with reduced learning rate
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(pruned_model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=finetune_epochs)

    for epoch in range(finetune_epochs):
        train_epoch(pruned_model, trainloader, optimizer, criterion)
        scheduler.step()

    acc_after_finetune = evaluate(pruned_model, testloader)
    return acc_after_finetune, acc_after_pruning, pruned_model, keep_idx


print('Pruning functions defined.')

Pruning functions defined.


## 8. Main Experiment Loop

Runs all methods at 20%, 30%, 40% sparsity — same protocol as CIFAR-10 paper experiment.

In [30]:
# #IF the code stops midway after saving the base model
# # 1. Re-instantiate the model exactly as before
# baseline_model = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)

# # 2. Load the weights back in
# path = os.path.join(SAVE_DIR, 'baseline_cifar100.pth')
# baseline_model.load_state_dict(torch.load(path, map_location=DEVICE))

# # 3. Set to evaluation mode (if you are testing)
# # OR keep in training mode if you want to resume training
# baseline_model.eval()

# print("baseline_model has been restored to its trained state.")

'#IF the code stops midway after saving the base model\n# 1. Re-instantiate the model exactly as before\nbaseline_model = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)\n\n# 2. Load the weights back in\npath = os.path.join(SAVE_DIR, \'baseline_cifar100.pth\')\nbaseline_model.load_state_dict(torch.load(path, map_location=DEVICE))\n\n# 3. Set to evaluation mode (if you are testing) \n# OR keep in training mode if you want to resume training\nbaseline_model.eval() \n\nprint("baseline_model has been restored to its trained state.")'

In [31]:
SPARSITY_LEVELS = [0.20, 0.30, 0.40]
METHODS = ['Spectral (AEI)', 'Hybrid (AEI×L2)',
           'L1 Norm', 'L2 Norm', 'SNIP', 'GraSP', 'Random']

results = defaultdict(dict)   # results[method][sparsity] = acc
keep_indices = defaultdict(dict)  # for spectral overlap analysis
overhead_log = {}

# ── Step 1: Compute AEI scores ONCE (expensive step) ─────────────────────────
print('Computing AEI scores...')
t0 = time.time()
R_scores, A_graph, fiedler_vec, overhead = get_aei_scores(
    baseline_model, activationloader)
overhead_log['Spectral (AEI)'] = overhead
print(f'AEI computation done in {time.time()-t0:.1f}s')
print(f'  Activation collection: {overhead["act_collect_ms"]:.0f}ms')
print(f'  Graph construction:    {overhead["graph_ms"]:.1f}ms')
print(f'  Eigendecomposition:    {overhead["eig_ms"]:.1f}ms')
print(f'  Total:                 {overhead["total_ms"]:.0f}ms')

# ── Step 2: Compute all other scores ─────────────────────────────────────────
print('\nComputing other method scores...')

t1 = time.time()
l1_scores = score_l1(baseline_model)
overhead_log['L1 Norm'] = {'total_ms': (time.time()-t1)*1000}

t1 = time.time()
l2_scores = score_l2(baseline_model)
overhead_log['L2 Norm'] = {'total_ms': (time.time()-t1)*1000}

t1 = time.time()
snip_scores = score_snip(baseline_model, activationloader)
overhead_log['SNIP'] = {'total_ms': (time.time()-t1)*1000}

t1 = time.time()
grasp_scores = score_grasp(baseline_model, activationloader)
overhead_log['GraSP'] = {'total_ms': (time.time()-t1)*1000}

hybrid_scores = score_hybrid(R_scores, baseline_model)
overhead_log['Hybrid (AEI×L2)'] = {
    'total_ms': overhead_log['L2 Norm']['total_ms'] + overhead['total_ms']}

scores_map = {
    'Spectral (AEI)':  R_scores,
    'Hybrid (AEI×L2)': hybrid_scores,
    'L1 Norm':         l1_scores,
    'L2 Norm':         l2_scores,
    'SNIP':            snip_scores,
    'GraSP':           grasp_scores,
    'Random':          None,   # computed fresh per trial
}

print('All scores computed.')
print('\nOverhead summary:')
for method, oh in overhead_log.items():
    print(f'  {method:<20}: {oh["total_ms"]:.0f}ms')

Computing AEI scores...
AEI computation done in 2.8s
  Activation collection: 1767ms
  Graph construction:    1068.5ms
  Eigendecomposition:    1.2ms
  Total:                 2841ms

Computing other method scores...
All scores computed.

Overhead summary:
  Spectral (AEI)      : 2841ms
  L1 Norm             : 1ms
  L2 Norm             : 0ms
  SNIP                : 759ms
  GraSP               : 1550ms
  Hybrid (AEI×L2)     : 2841ms


In [ ]:
# ── Step 3: Prune & Fine-tune all methods at all sparsities ──────────────────
print('\n' + '='*65)
print('PRUNING EXPERIMENTS — CIFAR-100')
print('='*65)

N_RANDOM_TRIALS = 5   # matches paper's comparison table (not 1000-trial distrib)

for method in METHODS:
    print(f'\n── {method} ──')
    for sparsity in SPARSITY_LEVELS:
        model_copy = copy.deepcopy(baseline_model)

        if method == 'Random':
            # Average over N_RANDOM_TRIALS
            trial_accs = []
            for trial in range(N_RANDOM_TRIALS):
                rand_scores = score_random(model_copy)
                acc, _, _, kidx = prune_and_finetune(
                    model_copy, rand_scores, sparsity,
                    num_classes=NUM_CLASSES)
                trial_accs.append(acc)
            mean_acc = np.mean(trial_accs)
            std_acc  = np.std(trial_accs)
            results[method][sparsity] = (mean_acc, std_acc)
            print(f'  Sparsity {int(sparsity*100)}%: '
                  f'{mean_acc:.2f}% ± {std_acc:.2f}%')
        else:
            scores = scores_map[method]
            acc, acc_before_ft, pruned_m, kidx = prune_and_finetune(
                model_copy, scores, sparsity, num_classes=NUM_CLASSES)
            results[method][sparsity] = acc
            keep_indices[method][sparsity] = kidx
            n_keep = len(kidx)
            flops  = count_flops(pruned_m) / 1e6
            print(f'  Sparsity {int(sparsity*100)}%: '
                  f'Before FT={acc_before_ft:.2f}% | '
                  f'After FT={acc:.2f}% | '
                  f'Filters kept={n_keep}/64 | FLOPs={flops:.2f}M')

            # Save pruned model
            save_name = f'{method.replace(" ","_").replace("×","x")}_{int(sparsity*100)}.pth'
            torch.save(pruned_m.state_dict(),
                       os.path.join(SAVE_DIR, save_name))

print('\nAll experiments complete.')


PRUNING EXPERIMENTS — CIFAR-100

── Spectral (AEI) ──
  Sparsity 20%: Before FT=42.12% | After FT=49.95% | Filters kept=51/64 | FLOPs=11.01M
  Sparsity 30%: Before FT=33.52% | After FT=49.26% | Filters kept=44/64 | FLOPs=9.75M
  Sparsity 40%: Before FT=28.87% | After FT=48.60% | Filters kept=38/64 | FLOPs=8.67M

── Hybrid (AEI×L2) ──
  Sparsity 20%: Before FT=43.29% | After FT=50.64% | Filters kept=51/64 | FLOPs=11.01M
  Sparsity 30%: Before FT=42.12% | After FT=50.06% | Filters kept=44/64 | FLOPs=9.75M
  Sparsity 40%: Before FT=42.12% | After FT=50.21% | Filters kept=38/64 | FLOPs=8.67M

── L1 Norm ──
  Sparsity 20%: Before FT=51.43% | After FT=51.46% | Filters kept=51/64 | FLOPs=11.01M
  Sparsity 30%: Before FT=51.44% | After FT=51.44% | Filters kept=44/64 | FLOPs=9.75M
  Sparsity 40%: Before FT=35.20% | After FT=50.12% | Filters kept=38/64 | FLOPs=8.67M

── L2 Norm ──
  Sparsity 20%: Before FT=51.43% | After FT=51.49% | Filters kept=51/64 | FLOPs=11.01M
  Sparsity 30%: Before FT=51

## 9. Results Tables

In [ ]:
print('='*65)
print('TABLE: Fine-tuned Accuracy on CIFAR-100 (SimpleCNN, conv2 filter pruning)')
print('='*65)
print(f'{"Method":<22} {"20%":>10} {"30%":>10} {"40%":>10}')
print('-'*55)

for method in METHODS:
    row = f'{method:<22}'
    for sparsity in SPARSITY_LEVELS:
        val = results[method][sparsity]
        if isinstance(val, tuple):
            row += f'  {val[0]:.2f}±{val[1]:.2f}'
        else:
            row += f'  {val:.2f}%  '
    print(row)

print('-'*55)
print(f'{"Baseline":<22}  {baseline_acc:.2f}%  (all sparsities)')

# ── FLOPs table ──────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('TABLE: FLOPs before and after pruning (SimpleCNN, CIFAR-100)')
print('='*55)
print(f'{"Sparsity":<12} {"Filters":>8} {"FLOPs":>10} {"Reduction":>12}')
print('-'*45)
baseline_flops = count_flops(baseline_model) / 1e6
print(f'{"Baseline":<12} {64:>8} {baseline_flops:>9.2f}M {"--":>12}')

for sparsity in SPARSITY_LEVELS:
    n_keep = max(1, int(64 * (1.0 - sparsity)))
    tmp = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=n_keep).to(DEVICE)
    flops = count_flops(tmp) / 1e6
    reduction = 100 * (1 - flops / baseline_flops)
    print(f'{int(sparsity*100):>3}%         {n_keep:>8} '
          f'{flops:>9.2f}M {reduction:>10.1f}%')

# ── Overhead table ───────────────────────────────────────────────────────────
print('\n' + '='*65)
print('TABLE: Pruning overhead (ms, one-time offline cost) at 30% sparsity')
print('='*65)
oh = overhead  # AEI overhead dict
print(f'{"Method":<22} {"Total":>8} {"Act.Collect":>12} {"Graph":>8} {"Eig":>8}')
print('-'*62)
print(f'{"Spectral (AEI)":<22} {oh["total_ms"]:>7.0f}ms '
      f'{oh["act_collect_ms"]:>10.0f}ms '
      f'{oh["graph_ms"]:>6.1f}ms '
      f'{oh["eig_ms"]:>6.1f}ms')
for method in ['L1 Norm', 'L2 Norm', 'SNIP', 'GraSP', 'Random']:
    t = overhead_log.get(method, {}).get('total_ms', 0)
    print(f'{method:<22} {t:>7.0f}ms {"--":>12} {"--":>8} {"--":>8}')

## 10. Spectral Overlap Analysis

The key diagnostic: where does each method prune on the AEI R-value spectrum?
This directly mirrors the MNIST/FashionMNIST analysis in the paper.

In [ ]:
def spectral_overlap_analysis(R_scores, keep_indices_dict, sparsity=0.30):
    """
    For each method, compute:
     - overlap with AEI-pruned filters
     - average R-percentile of pruned filters
    """
    F = len(R_scores)
    R_percentiles = scipy.stats.rankdata(R_scores) / F * 100

    # AEI pruned set (lowest R = structural periphery)
    aei_keep = set(keep_indices_dict['Spectral (AEI)'][sparsity].tolist())
    aei_prune = set(range(F)) - aei_keep

    print(f'\nSpectral Overlap Analysis at {int(sparsity*100)}% sparsity — CIFAR-100')
    print('='*70)
    print(f'{"Method":<22} {"Overlap w/ AEI":>16} {"Avg R-Pctile":>14} {"Interpretation"}')
    print('-'*70)

    overlap_data = {}
    for method in ['L1 Norm', 'L2 Norm', 'SNIP', 'GraSP', 'Random']:
        if method not in keep_indices_dict or sparsity not in keep_indices_dict[method]:
            continue
        m_keep  = set(keep_indices_dict[method][sparsity].tolist())
        m_prune = set(range(F)) - m_keep

        overlap = len(aei_prune & m_prune) / len(aei_prune) * 100
        avg_pctile = R_percentiles[list(m_prune)].mean() if m_prune else 0

        if avg_pctile < 35:
            interp = 'Targets periphery'
        elif avg_pctile < 55:
            interp = 'Slight bias toward peripheral'
        else:
            interp = 'Prunes central/critical filters'

        print(f'{method:<22} {overlap:>14.1f}% {avg_pctile:>13.1f} {interp}')
        overlap_data[method] = {'overlap': overlap, 'avg_pctile': avg_pctile}

    # AEI itself
    aei_pctile = R_percentiles[list(aei_prune)].mean()
    print(f'{"Spectral (AEI)":<22} {100.0:>14.1f}% {aei_pctile:>13.1f} Structural periphery (by design)')

    return overlap_data, R_percentiles


overlap_data, R_percentiles = spectral_overlap_analysis(
    R_scores, keep_indices, sparsity=0.30)

## 11. Visualizations

In [ ]:
def plot_spectral_histograms(R_scores, keep_indices_dict,
                              sparsity=0.30, save_path=None):
    """
    Mirrors the spectral histogram figure from the paper (Figure 10/11).
    Shows where each method prunes on the AEI R-value spectrum.
    """
    F       = len(R_scores)
    methods = ['Spectral (AEI)', 'Hybrid (AEI×L2)',
               'L1 Norm', 'L2 Norm', 'GraSP', 'SNIP', 'Random']

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    fig.suptitle(
        f'Where each method prunes on the spectral (AEI R-value) spectrum\n'
        f'CIFAR-100, {int(sparsity*100)}% sparsity',
        fontsize=13, fontweight='bold')

    all_idx = set(range(F))

    for ax, method in zip(axes, methods):
        if method not in keep_indices_dict or \
           sparsity not in keep_indices_dict[method]:
            ax.axis('off')
            continue

        keep  = set(keep_indices_dict[method][sparsity].tolist())
        prune = all_idx - keep

        R_keep  = R_scores[list(keep)]
        R_prune = R_scores[list(prune)]

        # Overlap with AEI
        aei_keep  = set(keep_indices_dict['Spectral (AEI)'][sparsity].tolist())
        aei_prune = all_idx - aei_keep
        overlap = len(aei_prune & prune) / max(len(aei_prune), 1) * 100
        avg_pctile = np.mean([R_percentiles[i] for i in prune]) if prune else 0

        color = '#2196F3' if method == 'Spectral (AEI)' else '#F44336'

        bins = np.linspace(R_scores.min(), R_scores.max(), 20)
        ax.hist(R_keep,  bins=bins, alpha=0.4, color='grey',  label='kept')
        ax.hist(R_prune, bins=bins, alpha=0.7, color=color,   label='pruned')

        ax.set_xlabel('R value (AEI score)', fontsize=8)
        ax.set_ylabel('Filter count', fontsize=8)

        title = method
        if method != 'Spectral (AEI)':
            title += f'\noverlap w/ AEI: {overlap:.1f}% | avg R-pctile: {avg_pctile:.1f}'
        ax.set_title(title, fontsize=8)
        ax.legend(fontsize=7)

    axes[-1].axis('off')   # 7 methods, 8 subplots
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


plot_spectral_histograms(
    R_scores, keep_indices, sparsity=0.30,
    save_path=os.path.join(SAVE_DIR, 'spectral_analysis_CIFAR100_30pct.pdf'))

In [ ]:
def plot_accuracy_comparison(results, baseline_acc, save_path=None):
    """
    Bar chart comparing all methods across sparsity levels.
    """
    sparsities = [20, 30, 40]
    methods_plot = ['Spectral (AEI)', 'Hybrid (AEI×L2)',
                    'L1 Norm', 'L2 Norm', 'SNIP', 'GraSP', 'Random']

    colors = ['#2196F3', '#9C27B0', '#FF9800', '#4CAF50',
              '#F44336', '#607D8B', '#795548']

    x = np.arange(len(sparsities))
    width = 0.11

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, (method, color) in enumerate(zip(methods_plot, colors)):
        accs = []
        errs = []
        for s in [0.20, 0.30, 0.40]:
            val = results[method][s]
            if isinstance(val, tuple):
                accs.append(val[0])
                errs.append(val[1])
            else:
                accs.append(val)
                errs.append(0)
        offset = (i - len(methods_plot)/2 + 0.5) * width
        ax.bar(x + offset, accs, width, label=method,
               color=color, alpha=0.85,
               yerr=errs if any(e>0 for e in errs) else None,
               capsize=3)

    ax.axhline(baseline_acc, color='black', linestyle='--',
               linewidth=1.5, label=f'Baseline ({baseline_acc:.2f}%)')

    ax.set_xlabel('Sparsity Level (%)', fontsize=12)
    ax.set_ylabel('Fine-tuned Accuracy (%)', fontsize=12)
    ax.set_title('CIFAR-100: Pruning Method Comparison (SimpleCNN)',
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{s}%' for s in sparsities])
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()


plot_accuracy_comparison(
    results, baseline_acc,
    save_path=os.path.join(SAVE_DIR, 'accuracy_comparison_CIFAR100.pdf'))

## 12. Multi-Seed Robustness (5 Seeds)

Matches the statistical robustness analysis from the FashionMNIST section of the paper.

In [ ]:
SEEDS = [42, 123, 456, 789, 1024]
SPARSITY_STATS = 0.40   # focus on 40% — the hardest case

seed_results = defaultdict(list)   # method -> list of accs across seeds

print(f'Running 5-seed robustness analysis at {int(SPARSITY_STATS*100)}% sparsity...')
print('(This takes a while — each seed retrains baseline + all methods)')

for seed in SEEDS:
    print(f'\n── Seed {seed} ──')
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Train fresh baseline for this seed
    m = SimpleCNN(num_classes=NUM_CLASSES, conv2_filters=64).to(DEVICE)
    train_model(m, epochs=TRAIN_EPOCHS, verbose=False)

    # Get AEI scores for this seed's model
    R_seed, _, _, _ = get_aei_scores(m, activationloader)
    l2_seed   = score_l2(m)
    l1_seed   = score_l1(m)
    snip_seed = score_snip(m, activationloader)
    try:
        grasp_seed = score_grasp(m, activationloader)
    except Exception:
        grasp_seed = score_l2(m)  # fallback
    hybrid_seed = score_hybrid(R_seed, m)

    seed_scores = {
        'Spectral (AEI)':  R_seed,
        'Hybrid (AEI×L2)': hybrid_seed,
        'L1 Norm':         l1_seed,
        'L2 Norm':         l2_seed,
        'SNIP':            snip_seed,
        'GraSP':           grasp_seed,
    }

    for method, sc in seed_scores.items():
        acc, _, _, _ = prune_and_finetune(
            m, sc, SPARSITY_STATS, num_classes=NUM_CLASSES)
        seed_results[method].append(acc)
        print(f'  {method:<22}: {acc:.2f}%')

print('\n' + '='*65)
print(f'5-SEED STATISTICS at {int(SPARSITY_STATS*100)}% sparsity — CIFAR-100')
print('='*65)
print(f'{"Method":<22} {"Mean":>8} {"Std":>8}')
print('-'*42)
for method in ['L2 Norm', 'L1 Norm', 'SNIP',
               'Hybrid (AEI×L2)', 'GraSP', 'Spectral (AEI)']:
    accs = seed_results[method]
    print(f'{method:<22} {np.mean(accs):>7.2f}% {np.std(accs):>7.2f}%')

## 13. Save All Results

In [ ]:
import json

# Convert results to JSON-serializable format
results_json = {}
for method, sparsity_dict in results.items():
    results_json[method] = {}
    for sparsity, val in sparsity_dict.items():
        if isinstance(val, tuple):
            results_json[method][str(sparsity)] = {'mean': val[0], 'std': val[1]}
        else:
            results_json[method][str(sparsity)] = val

summary = {
    'dataset':       'CIFAR-100',
    'architecture':  'SimpleCNN (Conv1:3→32, Conv2:32→64, FC1:4096→256, FC2:256→100)',
    'pruning_target':'Conv2 filters',
    'baseline_acc':  baseline_acc,
    'baseline_flops_M': count_flops(baseline_model)/1e6,
    'sparsity_levels': [0.20, 0.30, 0.40],
    'results':       results_json,
    'overhead_ms':   {k: v.get('total_ms', 0)
                      for k, v in overhead_log.items()},
    'seed_results_40pct': {
        k: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
        for k, v in seed_results.items()
    }
}

summary_path = os.path.join(SAVE_DIR, 'cifar100_results_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Results saved to {summary_path}')
print('\n=== EXPERIMENT COMPLETE ===')
print(f'Baseline accuracy: {baseline_acc:.2f}%')
print('\nFinal accuracy table (fine-tuned):')
print(f'{"Method":<22} {"20%":>8} {"30%":>8} {"40%":>8}')
print('-'*50)
for method in METHODS:
    row = f'{method:<22}'
    for s in SPARSITY_LEVELS:
        val = results[method][s]
        if isinstance(val, tuple):
            row += f'  {val[0]:.2f}'
        else:
            row += f'  {val:.2f}'
    print(row)